# Query was a quick attempt to see if we can mimick functionality of geefInvoervereisten
# We getting somewhere but result is not to be used
# Potentially finetune this further if we want to properly automate creation without relying on R.

In [10]:
import sqlite3
import pandas as pd

def geef_invoervereisten_kort(db_path, output_csv="invoervereisten_kort.csv"):
    conn = sqlite3.connect(db_path)
    
    # Eén grote query die alle Joins doet én de komma-gescheiden waarden berekent
    mega_query = """
    SELECT 
        'RBB v1' AS Versie,
        H.code AS Habitattype,
        C.Naam AS Criterium,
        I.Naam AS Indicator,
        B.Beoordeling_letterlijk AS Beoordeling,
        B.Kwaliteitsniveau,
        IB.Belang,
        B.Id AS BeoordelingID,
        CV.VoorwaardeID1 || IFNULL(' ' || CV.BewerkingOperator || ' ' || CV.VoorwaardeID2, '') AS Combinatie,
        VW.Id AS VoorwaardeID,
        VW.VoorwaardeNaam AS Voorwaarde,
        VW.Referentiewaarde,
        VW.Operator,
        VW.Maximumwaarde,
        AV.VariabeleNaam AS AnalyseVariabele,
        AV.Eenheid,
        TV.Naam AS TypeVariabele,
        L.Naam AS Invoertype,
        GROUP_CONCAT(DISTINCT LI.Waarde) AS Invoerwaarde,
        VW.TaxongroepId,
        TG.Omschrijving AS TaxongroepNaam,
        SG.Naam AS Studiegroepnaam,
        SG.LijstNaam AS Studielijstnaam,
        GROUP_CONCAT(DISTINCT SI.Waarde) AS Studiewaarde,
        SubAV.VariabeleNaam AS SubAnalyseVariabele,
        SubAV.Eenheid AS SubEenheid,
        TypeSub.Naam AS TypeSubVariabele,
        VW.SubReferentiewaarde,
        VW.SubOperator,
        SubL.Naam AS SubInvoertype,
        GROUP_CONCAT(DISTINCT SubLI.Waarde) AS SubInvoerwaarde
    FROM Indicator_habitat IH
    JOIN Habitattype H ON IH.HabitattypeId = H.Id
    JOIN Indicator_beoordeling IB ON IH.IndicatorId = IB.Id
    JOIN Indicator I ON IB.IndicatorId = I.Id
    JOIN Criterium C ON I.CriteriumId = C.Id
    JOIN Beoordeling B ON IB.Id = B.Indicator_beoordelingId
    JOIN CombinerenVoorwaarden CV ON B.Id = CV.BeoordelingId
    
    -- Hier doen we de "gather" (we splitsen Voorwaarde 1 en 2 naar aparte rijen)
    JOIN Voorwaarde VW ON VW.Id IN (CV.VoorwaardeID1, CV.VoorwaardeID2)
    
    -- Alle achterliggende details aanhaken
    LEFT JOIN Taxongroep TG ON VW.TaxongroepID = TG.Id
    LEFT JOIN AnalyseVariabele AV ON VW.AnalyseVariabeleID = AV.Id
    LEFT JOIN TypeVariabele TV ON AV.TypeVariabeleID = TV.Id
    LEFT JOIN Lijst L ON VW.InvoermaskerId = L.Id
    LEFT JOIN LijstItem LI ON L.Id = LI.LijstId
    LEFT JOIN Studiegroep SG ON VW.StudiegroepId = SG.Id
    LEFT JOIN StudieItem SI ON SG.Id = SI.StudiegroepId
    LEFT JOIN AnalyseVariabele SubAV ON VW.SubAnalyseVariabeleID = SubAV.Id
    LEFT JOIN TypeVariabele TypeSub ON SubAV.TypeVariabeleID = TypeSub.Id
    LEFT JOIN Lijst SubL ON VW.SubInvoermaskerId = SubL.Id
    LEFT JOIN LijstItem SubLI ON SubL.Id = SubLI.LijstId
    
    -- Zorg dat we per unieke vraag groeperen zodat GROUP_CONCAT netjes werkt
    GROUP BY H.Naam, C.Naam, I.Naam, B.Id, VW.Id
    """
    
    # 1. Voer de query uit in Pandas
    df = pd.read_sql_query(mega_query, conn)
    
    # 2. Exporteer direct naar CSV
    df.to_csv(output_csv, sep=';', index=False, na_rep='NA', encoding='utf-8-sig')
    
    conn.close()
    return df

# geef_invoervereisten_kort('LSVIHabitatTypes.sqlite')

In [11]:
test_pdf = geef_invoervereisten_kort('../input/LSVIHabitatTypes.sqlite')

In [5]:
test_pdf.shape

(2040, 32)

In [12]:
test_pdf[test_pdf['Habitattype'] == 'rbbah']

,Versie,Habitattype,Criterium,Indicator,Beoordeling,Kwaliteitsniveau,Belang,BeoordelingID,Combinatie,VoorwaardeID,...,Studiegroepnaam,Studielijstnaam,Studiewaarde,SubAnalyseVariabele,SubEenheid,TypeSubVariabele,SubReferentiewaarde,SubOperator,SubInvoertype,SubInvoerwaarde
989,RBB v1,rbbah,Structuur,bodemstructuur,A: geen breuklijnen (geleidelijk reliëf),2,zb,17,2135,2135,...,None,None,None,None,None,None,None,None,None,None
990,RBB v1,rbbah,Structuur,bodemstructuur,B: geen breuklijnen (geleidelijk reliëf),1,zb,18,2132,2132,...,None,None,None,None,None,None,None,None,None,None
991,RBB v1,rbbah,Structuur,horizontale structuur,A: vegetatievlek >= 10 m²,2,zb,237,2318,2318,...,None,None,None,None,None,None,None,None,None,None
992,RBB v1,rbbah,Structuur,horizontale structuur,B: vegetatievlek 1 - 10 m²,1,zb,238,2319,2319,...,None,None,None,None,None,None,None,None,None,None
993,RBB v1,rbbah,Structuur,ouderdomsstructuur Struikheide,B: 2 tot 3 stadia aanwezig,1,b,183,315,315,...,20pioniersstadium&40ontwikkelingsstadium&50cli...,ouderdomsstadia,"climaxstadium,degeneratiestadium,ontwikkelings...",None,None,None,None,None,None,None
994,RBB v1,rbbah,Structuur,ouderdomsstructuur Struikheide,A: alle stadia aanwezig,2,b,184,319,319,...,20pioniersstadium&40ontwikkelingsstadium&50cli...,ouderdomsstadia,"climaxstadium,degeneratiestadium,ontwikkelings...",None,None,None,None,None,None,None
995,RBB v1,rbbah,Verstoring,overige exoten,A: < 1%,2,b,121,1193,1193,...,None,None,None,None,None,None,None,None,None,None
996,RBB v1,rbbah,Verstoring,overige exoten,B: 1-10%,1,b,122,1202,1202,...,None,None,None,None,None,None,None,None,None,None
997,RBB v1,rbbah,Verstoring,vergrassing,B: 10-30%,1,zb,111,1723,1723,...,None,None,None,None,None,None,None,None,None,None
998,RBB v1,rbbah,Verstoring,vergrassing,A: < 10%,2,zb,112,1712,1712,...,None,None,None,None,None,None,None,None,None,None


In [9]:
test_pdf.Habitattype.unique()

array(['Actief hoogveen', 'Alkalisch laagveen',
       'Atlantische vastgelegde ontkalkte duinen (Calluno-Ulicetae)',
       'Atlantische zuurminnende beukenbossen met Ilex en soms ook Taxus in de ondergroei (Quercion robori-petraeae of Ilici-Fagenion)',
       'Beboste duinen van het Atlantische, Continentale en Boreale kustgebied',
       'Beukenbossen van het type Asperulo-Fagetum.',
       'Beukenbossen van het type Luzulo-Fagetum', 'Droge Europese heide',
       'Droge half-natuurlijke graslanden en struikvormende faciës op kalkhoudende bodems (Festuco Brometalia)',
       'Duinen met Hyppophae rhamnoides',
       'Duinen met Salix repens ssp. argentea (Salicion arenaria)',
       'Dystrofe natuurlijke poelen en meren',
       'Embryonale wandelende duinen',
       'Gemengde oeverformaties met Quercus robur, Ulmus laevis en Ulmus minor, Fraxinus excelsior of Fraxinus angustifolia, langs de grote rivieren (Ulmenion minoris)',
       'Grasland met Molinia op kalkhoudende, venige of 